# Retrieval Comparison

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.rag.config import load_config

from src.rag.embedding.factory import EmbeddingFactory
from src.rag.preprocessing.processor import TextProcessor

from src.rag.vector_store.faiss import FAISSVectorStore

from src.rag.retrieval.bm25 import BM25Retriever
from src.rag.retrieval.embedding import EmbeddingRetriever
from src.rag.retrieval.hybrid import HybridRetriever


In [3]:
config = load_config("../configs/rag.yaml")

config


{'retrieval': {'type': 'hybrid',
  'top_k': 5,
  'hybrid': {'bm25_weight': 0.3, 'embedding_weight': 0.7}},
 'embedding': {'provider': 'sentence_transformer',
  'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'}}

In [4]:
processor = TextProcessor()


embedding_model = EmbeddingFactory.create(
    provider=config["embedding"]["provider"],
    model_name=config["embedding"]["model"]
)


Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 16474.87it/s]


In [5]:
# Load chunked FAISS index

faiss_path = Path(
    "../data/indexes/product_comments_embedding"
)

vector_store = FAISSVectorStore()

vector_store.load(
    faiss_path
)

vector_store.documents.head()


,id,product_id,body,rate,search_text
0,14144758,1075274,عالیه مخصوصا طرحش عکس مجموعه هری پاترم رو میزا...,5.0,هری پاتر عالیه مخصوصا طرحش عکس مجموعه هری پاتر...
1,41782279,6081008,توجه فرمایید که عکس اول واسه ۲ هفته پیشه و عکس...,3.0,روغن ریش توجه فرمایید که عکس اول واسه ۲ هفته پ...
2,49569443,10545754,متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.بالای ۴...,1.0,تقلبی متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.ب...
3,43932524,4153832,تصمیم خرید کنسول برای منِ 32 ساله با هزینه شخص...,5.0,یک نظر بی اغراق! تصمیم خرید کنسول برای منِ 32 ...
4,21396693,2185657,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...,3.0,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...


In [7]:
# Load BM25 index

bm25_path = Path(
    "../data/indexes/product_comments_bm25"
)

bm25_retriever = BM25Retriever(
    processor=processor
)

bm25_retriever.load(
    bm25_path
)


In [8]:
# Create embedding retriever from loaded chunked FAISS

embedding_retriever = EmbeddingRetriever.__new__(
    EmbeddingRetriever
)

embedding_retriever.documents = vector_store.documents
embedding_retriever.embedding_model = embedding_model
embedding_retriever.processor = processor
embedding_retriever.vector_store = vector_store


In [9]:
# Create hybrid retriever

hybrid_config = config["retrieval"]["hybrid"]

hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    embedding_retriever=embedding_retriever,
    bm25_weight=hybrid_config["bm25_weight"],
    embedding_weight=hybrid_config["embedding_weight"]
)


In [10]:
query = "آیا برای پوست چرب مناسب است؟"

top_k = 5


## BM25

In [11]:
bm25_results = bm25_retriever.retrieve(
    query,
    top_k=top_k
)

bm25_results[
    ["body", "rate", "score"]
]


,body,rate,score
2890141,عالی برای پوست چرب,5.0,23.807002
2055701,مناسب پوست چرب,5.0,23.769173
2044511,مناسب پوست چرب,3.0,22.979941
3486635,برای پوست چرب خوب,4.0,22.013559
2061392,برای پوست چرب مناسب نیست,4.0,21.742143


## Embedding

In [12]:
embedding_results = embedding_retriever.retrieve(
    query,
    top_k=top_k
)

embedding_results[
    ["body", "rate", "score"]
]


,body,rate,score
4560,برای پوست چرب مناسب است.,5.0,0.937525
3895,برای پوست چرب مناسب است,5.0,0.927969
5910,برای پوست چرب قابل‌قبول,4.0,0.927583
1445,برای پوست چرب مناسبه.,3.0,0.925952
5915,برای پوست چرب مناسب هست,4.0,0.920723


## Hybrid

In [13]:
hybrid_results = hybrid_retriever.retrieve(
    query,
    top_k=top_k
)

hybrid_results


,body,bm25_score,embedding_score,score
8,برای پوست چرب مناسبه,0.268850,0.000004,0.080658
0,برای پوست های چرب مناسب است,NaN,0.220402,NaN
1,برای پوست های چرب مناسب است.,NaN,0.216601,NaN
2,برای پوست چرب خوب,0.477947,NaN,NaN
3,برای پوست چرب قابل‌قبول,NaN,0.626612,NaN
